In [22]:
import os
import sqlite3
from sre_parse import State
from typing import Annotated, TypedDict, Literal
from typing import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
import json
import pprint

from langgraph.graph.state import Checkpointer



# define model
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.0,
    base_url = os.getenv("base_url"),
    api_key = os.getenv("api_key"),
    max_tokens=None,
    max_retries=2,
)

class PersonalAssistantState(TypedDict):
    # short memory
    messages: Annotated[list, add_messages]

    #long memory
    user_name: str
    user_id: str
    learned_facts: [list, lambda old, new: old + new]
    preferences: dict

    # meta data
    session_id: str
    message_count: Annotated[int, lambda old, new: old + new]


# node
def chat_node(state: PersonalAssistantState) -> dict:
    messages = state.get("messages", [])
    user_name = state.get("user_name", "未知用户")
    user_id = state.get("user_id", "unknown")
    facts = state.get("learned_facts", [])
    preferences = state.get("preferences", {})

    system_template = f"""
        你是一个私人助理，你的服务用户是{user_name}，他的user_id为{user_id}
        你已知的事实是：{[fact + "\n" for fact in facts]}
        和用户的偏好：{[f"{k}:{v}\n" for k, v in preferences.items()]}
    """

    full_messages = [SystemMessage(content=system_template)] + messages[-10:]

    response = model.invoke(full_messages)

    return {
        "messages": [response],
        "message_count": 1,
    }

def extract_learnings_node(state: PersonalAssistantState) -> dict:
    contents = state["messages"][-2:]
    prompt = f"""
    从以下对话内容中，提取出有效的用户偏好和事实，并以严格的json格式返回, 不要出现任何其他的字符，包括“json”什么的。
    对话内容:{[str(content) for content in contents]}.
    返回JSON：
    {{
        "facts": ["fact1", "fact2"],
        "preferences": {{"key1": "value"}}
    }}
    """

    response = model.invoke([HumanMessage(content=prompt)])
    
    #Test
    print(response.content)
    
    
    extracted = json.loads(response.content)

    return {
        "facts": extracted["facts"],
        "preferences": extracted["preferences"]
    }

def should_extract(state: PersonalAssistantState) -> str:
    contents = state.get("messages", [])[-2:]
    prompt = f"""
        从以下对话内容中，判断是否有有效的用户偏好和事实.
        对话内容：{[str(content) for content in contents]}
        返回严格的true或false
    
    """
    response = model.invoke([HumanMessage(content=prompt)])
    if "true" in response.content.lower():
        return "extract"
    return "end"

# create graph

memory_graph = StateGraph(PersonalAssistantState)

memory_graph.add_node("chat", chat_node)
memory_graph.add_node("extract_learnings_node", extract_learnings_node)

memory_graph.add_edge(START, "chat")
memory_graph.add_conditional_edges(
    "chat",
    should_extract,
    {
        "extract": "extract_learnings_node",
        "end": END,
    },
)
memory_graph.add_edge("extract_learnings_node", END)

# use SQLiteCheckpoint 
checkpointer = SqliteSaver(sqlite3.connect("personal_assistant.db", check_same_thread=False))

# compile
app = memory_graph.compile(checkpointer=checkpointer)

# Test
config = {"configurable": {"thread_id": "user-alice"}}

# 给app.invoke传递config参数，比如设置thread_id（已在config变量里）。应如下方式给出：
result = app.invoke({"messages": [HumanMessage(content="Hello")]}, config=config)
import pprint
pprint.pprint(result)
result = app.invoke({"messages": [HumanMessage(content="what did I said")]}, config=config)
pprint.pprint(result)
result = app.invoke({"messages": [HumanMessage(content="I like eating chicken")]}, config=config)
pprint.pprint(result)
result = app.invoke({"messages": [HumanMessage(content="Do you know what I'd like to eat?")]}, config=config)
pprint.pprint(result)


{'message_count': 6,
 'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='7c587dec-0f04-4c19-b2af-8036d7e61fd1'),
              AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 54, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DOnaCCxRNCAlwOfai9OOGasFvh0wC', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3a7e-28cd-7a32-b15e-ce7014b5e6f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 10, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

In [31]:
"""
实战：交互式内容审核工作流
功能：
- 自动内容分析
- 流式输出分析进度
- 多级人工审核（Interrupts）
- 历史追溯（Time · Travel）
- 可撤销决策
"""

from langgraph.types import interrupt
import time


class ContentReviewState(TypedDict):
    # 待审内容 / 对话消息
    messages: Annotated[list, add_messages]
    # 人工审核时外部传入的操作（approve / escalate / reject）
    interrupt_operation: str
    # 分析结果
    toxicity: float
    spam: float
    quality: float
    # AI 决策
    auto_decision: str
    ai_reason: str
    # 处理步骤
    processing_steps: list
    # 审核相关
    review_level: int
    reviewer_decision: list
    final_decision: str


def analyze(state: ContentReviewState) -> dict:
    """自动内容分析：给出粗略的 toxicity / spam / quality 评分。"""
    messages = state.get("messages", [])
    last_msg = messages[-1] if messages else ""
    text = str(last_msg)

    # 非严格规则，只是示例：包含敏感词就提高 toxicity，包含链接就提高 spam
    lower = text.lower()
    toxicity = 0.9 if any(w in lower for w in ["hate", "violence", "kill", "destroy"]) else 0.1
    spam = 0.8 if "http" in lower or "www" in lower else 0.1
    quality = 0.9 if len(text) > 10 else 0.3

    print(f"分析结果 - 毒性: {toxicity}, 垃圾邮件: {spam}, 质量: {quality}")

    return {
        "toxicity": toxicity,
        "spam": spam,
        "quality": quality,
        "processing_steps": ["自动分析完成"],
    }


def ai_recommendation(state: ContentReviewState) -> dict:
    """调用 LLM 做自动初判，给出 approve / reject 建议。"""
    content = state.get("messages", [])[-1]  # 简化：最后一条消息是要审核的内容
    toxicity = state.get("toxicity", 0.1)
    spam = state.get("spam", 0.1)
    
    # 简化的决策逻辑：如果毒性或垃圾邮件评分高，就拒绝
    if toxicity > 0.5 or spam > 0.5:
        auto_decision = "reject"
        ai_reason = f"内容被标记为高风险 (毒性: {toxicity}, 垃圾邮件: {spam})"
    else:
        auto_decision = "approve"
        ai_reason = f"内容通过初步检查 (毒性: {toxicity}, 垃圾邮件: {spam})"

    steps = state.get("processing_steps", []) + [f"AI 初判: {auto_decision}"]

    print(f"AI 初判结果: {auto_decision} - {ai_reason}")

    return {
        "auto_decision": auto_decision,
        "ai_reason": ai_reason,
        "processing_steps": steps,
    }


def level1_review(state: ContentReviewState) -> dict:
    """一级人工审核：通过 interrupt 停下来等待人工决策。"""
    print("进入一级人工审核...")
    
    decision = interrupt(
        {
            "type": "level1_review",
            "content": state.get("messages", [])[-1],
            "scores": {
                "toxicity": state.get("toxicity", ""),
                "spam": state.get("spam", ""),
                "quality": state.get("quality", ""),
            },
            "options": ["approve", "escalate", "reject"],
        }
    )

    steps = state.get("processing_steps", []) + [f"一级审核完成: {decision}"]

    return {
        "review_level": 1,
        "interrupt_operation": decision,
        "reviewer_decision": [
            {
                "level": 1,
                "decision": decision,
                "timestamp": time.time(),
            }
        ],
        "processing_steps": steps,
    }


def level2_review(state: ContentReviewState) -> dict:
    """二级人工审核（通常在一级选择 escalate 时触发）。"""
    print("进入二级人工审核...")
    
    decision = interrupt(
        {
            "type": "level2_review",
            "content": state.get("messages", [])[-1],
            "scores": {
                "toxicity": state.get("toxicity", ""),
                "spam": state.get("spam", ""),
                "quality": state.get("quality", ""),
            },
            "options": ["approve", "reject", "flag"],
        }
    )

    steps = state.get("processing_steps", []) + [f"二级审核完成: {decision}"]

    return {
        "review_level": 2,
        "interrupt_operation": decision,
        "final_decision": decision,
        "processing_steps": steps,
    }


def route_after_ai(state: ContentReviewState) -> str:
    """根据 AI 初判结果决定是否进入人工审核。"""
    decision = state.get("auto_decision", "approve").lower()
    print(f"AI 路由决策: {decision}")
    if decision == "reject":
        return "human_review"  # 进入人工审核
    return "approve"  # 直接通过


def route_after_level1(state: ContentReviewState) -> str:
    """一级审核后的路由：通过 / 升级 / 拒绝。"""
    op = state.get("interrupt_operation", "").lower()
    print(f"一级审核路由决策: {op}")
    if op == "escalate":
        return "escalate"  # 升级到二级审核
    if op == "reject":
        return "reject"
    return "approve"


def route_after_level2(state: ContentReviewState) -> str:
    """二级审核后的路由：拒绝或标记。"""
    op = state.get("interrupt_operation", "").lower()
    print(f"二级审核路由决策: {op}")
    if op == "reject":
        return "reject"
    return "flag"


# 构建内容审核图
review_graph = StateGraph(ContentReviewState)

review_graph.add_node("analyze", analyze)
review_graph.add_node("ai_recommendation", ai_recommendation)
review_graph.add_node("level1_review", level1_review)
review_graph.add_node("level2_review", level2_review)

review_graph.add_edge(START, "analyze")
review_graph.add_edge("analyze", "ai_recommendation")

# AI 初判后：approve 直接结束，reject 进入一级人工审核
review_graph.add_conditional_edges(
    "ai_recommendation",
    route_after_ai,
    {
        "approve": END,
        "human_review": "level1_review",
    },
)

# 一级审核：approve / reject 结束，escalate 进入二级审核
review_graph.add_conditional_edges(
    "level1_review",
    route_after_level1,
    {
        "approve": END,
        "reject": END,
        "escalate": "level2_review",
    },
)

# 二级审核：reject / flag 都结束（你可以根据业务调整）
review_graph.add_conditional_edges(
    "level2_review",
    route_after_level2,
    {
        "reject": END,
        "flag": END,
    },
)

review_app = review_graph.compile()

In [ ]:
# ==================== 新增代码开始 ====================
# 为内容审核工作流添加检查点支持
review_checkpointer = SqliteSaver(sqlite3.connect("content_review.db", check_same_thread=False))
review_app = review_graph.compile(checkpointer=review_checkpointer, interrupt_before=["level1_review", "level2_review"])

# 测试内容审核工作流
print("=== 内容审核工作流测试 ===")

# 测试案例1：正常内容（应该直接通过）
print("\n--- 测试案例1：正常内容 ---")
config1 = {"configurable": {"thread_id": "review-thread-1-new"}}
try:
    result1 = review_app.invoke(
        {"messages": [HumanMessage(content="今天天气真好，适合出去散步")]}, 
        config=config1
    )
    print("正常内容处理结果：")
    pprint.pprint(result1)
except Exception as e:
    print(f"处理正常内容时出错: {e}")

# 测试案例2：可疑内容（需要人工审核）
print("\n--- 测试案例2：可疑内容 ---")
config2 = {"configurable": {"thread_id": "review-thread-2-new"}}
try:
    result2 = review_app.invoke(
        {"messages": [HumanMessage(content="I hate this stupid system, it should be destroyed")]}, 
        config=config2
    )
    print("可疑内容处理结果（应该在一级审核处中断）：")
    pprint.pprint(result2)
    
    # 查看当前状态
    print("\n当前工作流状态：")
    current_state = review_app.get_state(config2)
    print(f"下一步节点: {current_state.next}")
    print(f"处理步骤: {current_state.values.get('processing_steps', [])}")
    print(f"毒性评分: {current_state.values.get('toxicity', 'N/A')}")
    print(f"AI 决策: {current_state.values.get('auto_decision', 'N/A')}")
    
    # 正确的方式：使用 None 输入来继续中断的工作流
    print("\n--- 模拟一级审核决策：升级到二级审核 ---")
    # 更新状态以模拟人工审核决策
    review_app.update_state(config2, {"interrupt_operation": "escalate"})
    
    # 继续执行工作流
    result2_continue = review_app.invoke(None, config=config2)
    print("升级后的结果（应该在二级审核处中断）：")
    pprint.pprint(result2_continue)
    
    # 查看升级后的状态
    print("\n升级后的工作流状态：")
    current_state2 = review_app.get_state(config2)
    print(f"下一步节点: {current_state2.next}")
    print(f"处理步骤: {current_state2.values.get('processing_steps', [])}")
    
    # 模拟二级审核决策：拒绝
    print("\n--- 模拟二级审核决策：拒绝 ---")
    review_app.update_state(config2, {"interrupt_operation": "reject"})
    result2_final = review_app.invoke(None, config=config2)
    print("最终审核结果：")
    pprint.pprint(result2_final)
    
except Exception as e:
    print(f"处理可疑内容时出错: {e}")

# 测试案例3：垃圾邮件内容
print("\n--- 测试案例3：垃圾邮件内容 ---")
config3 = {"configurable": {"thread_id": "review-thread-3-new"}}
try:
    result3 = review_app.invoke(
        {"messages": [HumanMessage(content="Visit http://spam-site.com for amazing deals! Click now!")]}, 
        config=config3
    )
    print("垃圾邮件内容处理结果（应该在一级审核处中断）：")
    pprint.pprint(result3)
    
    # 查看状态
    current_state3 = review_app.get_state(config3)
    print(f"下一步节点: {current_state3.next}")
    print(f"垃圾邮件评分: {current_state3.values.get('spam', 'N/A')}")
    
    # 模拟一级审核：直接拒绝
    print("\n--- 模拟一级审核决策：直接拒绝 ---")
    review_app.update_state(config3, {"interrupt_operation": "reject"})
    result3_final = review_app.invoke(None, config=config3)
    print("垃圾邮件最终处理结果：")
    pprint.pprint(result3_final)
    
except Exception as e:
    print(f"处理垃圾邮件内容时出错: {e}")

print("\n=== 内容审核工作流测试完成 ===")
# ==================== 新增代码结束 ====================

=== 内容审核工作流测试 ===

--- 测试案例1：正常内容 ---
分析结果 - 毒性: 0.1, 垃圾邮件: 0.1, 质量: 0.9
AI 初判结果: approve - 内容通过初步检查 (毒性: 0.1, 垃圾邮件: 0.1)
AI 路由决策: approve
正常内容处理结果：
{'ai_reason': '内容通过初步检查 (毒性: 0.1, 垃圾邮件: 0.1)',
 'auto_decision': 'approve',
 'messages': [HumanMessage(content='今天天气真好，适合出去散步', additional_kwargs={}, response_metadata={}, id='a8845909-daf6-4f2b-821a-df9a242eb5ad')],
 'processing_steps': ['自动分析完成', 'AI 初判: approve'],
 'quality': 0.9,
 'spam': 0.1,
 'toxicity': 0.1}

--- 测试案例2：可疑内容 ---
分析结果 - 毒性: 0.9, 垃圾邮件: 0.1, 质量: 0.9
AI 初判结果: reject - 内容被标记为高风险 (毒性: 0.9, 垃圾邮件: 0.1)
AI 路由决策: reject
可疑内容处理结果（应该在一级审核处中断）：
{'ai_reason': '内容被标记为高风险 (毒性: 0.9, 垃圾邮件: 0.1)',
 'auto_decision': 'reject',
 'messages': [HumanMessage(content='I hate this stupid system, it should be destroyed', additional_kwargs={}, response_metadata={}, id='6f042b03-9d8d-4fb7-affc-ba8970b883ff')],
 'processing_steps': ['自动分析完成', 'AI 初判: reject'],
 'quality': 0.9,
 'spam': 0.1,
 'toxicity': 0.9}

当前工作流状态：
下一步节点: ('level1_review',)
处理步骤

In [ ]:
# ==================== 新增代码开始 ====================
# 演示时间旅行功能（Time Travel）
print("\n=== 时间旅行功能演示 ===")

# 首先创建一个有多轮对话的会话
print("\n--- 创建多轮对话会话 ---")
time_travel_config = {"configurable": {"thread_id": "time-travel-demo"}}

# 第一轮对话
print("第一轮：用户介绍自己")
result_t1 = app.invoke(
    {
        "messages": [HumanMessage(content="你好，我叫张三，我是一名软件工程师")],
        "user_name": "张三",
        "user_id": "zhangsan_001"
    }, 
    time_travel_config
)

# 第二轮对话
print("\n第二轮：用户表达偏好")
result_t2 = app.invoke(
    {"messages": [HumanMessage(content="我喜欢喝咖啡，特别是拿铁")]}, 
    time_travel_config
)

# 第三轮对话
print("\n第三轮：用户询问推荐")
result_t3 = app.invoke(
    {"messages": [HumanMessage(content="你能推荐一些适合我的饮品吗？")]}, 
    time_travel_config
)

# 查看完整的对话历史
print("\n--- 查看对话历史 ---")
history = app.get_state_history(time_travel_config)
states = list(history)
print(f"总共有 {len(states)} 个状态快照")

for i, state in enumerate(states):
    print(f"\n状态 {i+1} (配置: {state.config}):")
    print(f"  消息数量: {len(state.values.get('messages', []))}")
    print(f"  学习到的事实: {state.values.get('learned_facts', [])}")
    print(f"  用户偏好: {state.values.get('preferences', {})}")
    if state.values.get('messages'):
        last_msg = state.values['messages'][-1]
        print(f"  最后一条消息: {last_msg.content[:50]}...")

# 时间旅行：回到第二个状态
print("\n--- 时间旅行：回到第二个状态 ---") 
if len(states) >= 2:
    target_state = states[1]  # 第二个状态
    print(f"回到状态: {target_state.config}")
    
    # 从这个状态继续对话（创建分支）
    branch_config = {"configurable": {"thread_id": "time-travel-branch"}}
    
    # 使用 update_state 来设置初始状态
    app.update_state(branch_config, target_state.values)
    
    # 从分支点继续新的对话
    print("\n从分支点开始新对话：")
    branch_result = app.invoke(
        {"messages": [HumanMessage(content="其实我更喜欢茶，不是咖啡")]}, 
        branch_config
    )
    
    print("分支对话结果：")
    final_state = app.get_state(branch_config)
    print(f"  学习到的事实: {final_state.values.get('learned_facts', [])}")
    print(f"  用户偏好: {final_state.values.get('preferences', {})}")

print("\n=== 时间旅行功能演示完成 ===")
# ==================== 新增代码结束 ====================


=== 时间旅行功能演示 ===

--- 创建多轮对话会话 ---
第一轮：用户介绍自己

第二轮：用户表达偏好
{
    "facts": [
        "用户喜欢喝咖啡",
        "用户特别喜欢拿铁"
    ],
    "preferences": {
        "preferred_coffee_type": "拿铁"
    }
}

第三轮：用户询问推荐
{
    "facts": [
        "用户的名字是张三",
        "用户喜欢拿铁"
    ],
    "preferences": {
        "推荐饮品": [
            "香草拿铁",
            "焦糖拿铁",
            "摩卡咖啡",
            "杏仁拿铁",
            "杏仁奶拿铁"
        ],
        "适合人群": "不喜欢牛奶的人"
    }
}

--- 查看对话历史 ---
总共有 33 个状态快照

状态 1 (配置: {'configurable': {'thread_id': 'time-travel-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f12c413-5537-64e0-801f-931ba21c893e'}}):
  消息数量: 18
  学习到的事实: []
  用户偏好: {'推荐饮品': ['香草拿铁', '焦糖拿铁', '摩卡咖啡', '杏仁拿铁', '杏仁奶拿铁'], '适合人群': '不喜欢牛奶的人'}
  最后一条消息: 当然可以，张三！考虑到你喜欢拿铁，这里有一些适合你的饮品推荐：

1. **香草拿铁**：经典拿铁加...

状态 2 (配置: {'configurable': {'thread_id': 'time-travel-demo', 'checkpoint_ns': '', 'checkpoint_id': '1f12c413-375b-6d6f-801e-71c0fa361018'}}):
  消息数量: 18
  学习到的事实: []
  用户偏好: {'preferred_coffee_type': '拿铁'}
  最后一条消息:

In [34]:
# ==================== 新增代码开始 ====================
# 内存管理和数据持久化演示
print("\n=== 内存管理和数据持久化演示 ===")

# 1. 查看所有持久化的会话
print("\n--- 查看数据库中的会话信息 ---")
try:
    # 连接到个人助理数据库
    conn = sqlite3.connect("personal_assistant.db")
    cursor = conn.cursor()
    
    # 查看检查点表结构
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"个人助理数据库中的表: {[table[0] for table in tables]}")
    
    # 查看检查点数据
    cursor.execute("SELECT thread_id, checkpoint_id, step FROM checkpoints LIMIT 10;")
    checkpoints = cursor.fetchall()
    print(f"检查点数据样例: {checkpoints}")
    
    conn.close()
    
    # 连接到内容审核数据库
    conn2 = sqlite3.connect("content_review.db")
    cursor2 = conn2.cursor()
    
    cursor2.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables2 = cursor2.fetchall()
    print(f"内容审核数据库中的表: {[table[0] for table in tables2]}")
    
    conn2.close()
    
except Exception as e:
    print(f"查看数据库时出错: {e}")

# 2. 演示会话恢复功能
print("\n--- 演示会话恢复功能 ---")
recovery_config = {"configurable": {"thread_id": "recovery-test"}}

# 创建一个新会话
print("创建新会话并添加一些数据...")
app.invoke(
    {
        "messages": [HumanMessage(content="我是李四，我喜欢运动")],
        "user_name": "李四",
        "user_id": "lisi_001"
    }, 
    recovery_config
)

# 模拟程序重启后恢复会话
print("\n模拟程序重启后恢复会话...")
recovered_state = app.get_state(recovery_config)
print("恢复的会话状态:")
print(f"  用户名: {recovered_state.values.get('user_name', '未知')}")
print(f"  用户ID: {recovered_state.values.get('user_id', '未知')}")
print(f"  消息数量: {len(recovered_state.values.get('messages', []))}")
print(f"  学习到的事实: {recovered_state.values.get('learned_facts', [])}")

# 继续之前的对话
print("\n继续之前的对话...")
continued_result = app.invoke(
    {"messages": [HumanMessage(content="你还记得我喜欢什么吗？")]}, 
    recovery_config
)

# 3. 清理和维护功能
print("\n--- 内存清理和维护功能 ---")

def cleanup_old_sessions(days_old=30):
    """清理超过指定天数的旧会话"""
    import time
    cutoff_time = time.time() - (days_old * 24 * 60 * 60)
    
    try:
        conn = sqlite3.connect("personal_assistant.db")
        cursor = conn.cursor()
        
        # 这里只是演示，实际的清理逻辑会更复杂
        cursor.execute("SELECT COUNT(*) FROM checkpoints")
        total_checkpoints = cursor.fetchone()[0]
        print(f"当前总检查点数量: {total_checkpoints}")
        
        conn.close()
        print("清理功能演示完成（实际清理需要根据具体需求实现）")
        
    except Exception as e:
        print(f"清理时出错: {e}")

cleanup_old_sessions()

# 4. 内存使用统计
print("\n--- 内存使用统计 ---")
def get_memory_stats():
    """获取内存使用统计信息"""
    try:
        conn = sqlite3.connect("personal_assistant.db")
        cursor = conn.cursor()
        
        # 统计不同thread_id的数量
        cursor.execute("SELECT COUNT(DISTINCT thread_id) FROM checkpoints")
        unique_threads = cursor.fetchone()[0]
        
        # 统计总检查点数量
        cursor.execute("SELECT COUNT(*) FROM checkpoints")
        total_checkpoints = cursor.fetchone()[0]
        
        conn.close()
        
        return {
            "unique_sessions": unique_threads,
            "total_checkpoints": total_checkpoints,
            "avg_checkpoints_per_session": total_checkpoints / max(unique_threads, 1)
        }
        
    except Exception as e:
        print(f"获取统计信息时出错: {e}")
        return {}

stats = get_memory_stats()
print("内存使用统计:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print("\n=== LangGraph 内存组件实战演示完成 ===")
print("\n总结:")
print("1. ✅ 个人助理系统 - 短期记忆（对话历史）+ 长期记忆（用户偏好和事实）")
print("2. ✅ 内容审核工作流 - 多级人工审核 + 中断机制")
print("3. ✅ 时间旅行功能 - 状态回溯和分支对话")
print("4. ✅ 数据持久化 - SQLite 检查点存储")
print("5. ✅ 会话管理 - 恢复、清理和统计功能")
# ==================== 新增代码结束 ====================


=== 内存管理和数据持久化演示 ===

--- 查看数据库中的会话信息 ---
个人助理数据库中的表: ['checkpoints', 'writes']
查看数据库时出错: no such column: step

--- 演示会话恢复功能 ---
创建新会话并添加一些数据...
{
    "facts": ["用户的名字是李四", "李四喜欢运动"],
    "preferences": {}
}

模拟程序重启后恢复会话...
恢复的会话状态:
  用户名: 李四
  用户ID: lisi_001
  消息数量: 10
  学习到的事实: []

继续之前的对话...
{
    "facts": ["用户的名字是李四", "用户喜欢运动"],
    "preferences": {}
}

--- 内存清理和维护功能 ---
当前总检查点数量: 102
清理功能演示完成（实际清理需要根据具体需求实现）

--- 内存使用统计 ---
内存使用统计:
  unique_sessions: 4
  total_checkpoints: 102
  avg_checkpoints_per_session: 25.5

=== LangGraph 内存组件实战演示完成 ===

总结:
1. ✅ 个人助理系统 - 短期记忆（对话历史）+ 长期记忆（用户偏好和事实）
2. ✅ 内容审核工作流 - 多级人工审核 + 中断机制
3. ✅ 时间旅行功能 - 状态回溯和分支对话
4. ✅ 数据持久化 - SQLite 检查点存储
5. ✅ 会话管理 - 恢复、清理和统计功能
